In [ ]:
# main.ipynb
import numpy as np
import casadi as ca
from ocp_formulation import setup_ocp
from utils import unpack_solution, plot_trajectory

# Problem parameters
gear = 2           # selected gear for entire trajectory
dt = 0.1           # initial guess for time step
N = 50             # number of shooting intervals
use_soft_track = False  # use hard constraints for strict satisfaction

# Setup and solve the OCP
solver, nlp, integrator = setup_ocp(gear, dt, N, use_soft_track=use_soft_track)

# Initial guess for the decision variables
nx, nu = 7, 3
w0 = np.zeros(nlp['x'].shape[0])
w0[-1] = 7.0  # Initial guess for final time T

# Bounds on decision variables
lbw = [-ca.inf] * (nlp['x'].shape[0] - 1) + [1.0]  # minimum allowed final time
ubw = [ca.inf] * (nlp['x'].shape[0] - 1) + [20.0]  # maximum allowed final time

# Bounds on constraints (enforce equality constraints strictly)
lbg = [0] * nlp['g'].shape[0]
ubg = [0] * nlp['g'].shape[0]

# Solve the NLP
solution = solver(x0=w0, lbx=lbw, ubx=ubw, lbg=lbg, ubg=ubg)

# Unpack and visualize the solution
X_opt, U_opt, T_opt = unpack_solution(solution['x'], N, nx=nx, nu=nu)

# Plot the trajectory
plot_trajectory(X_opt, T_opt, track=True, title='Optimal Car Obstacle Avoidance Trajectory')
